# Diff assigned parameters

This might be useful if you're getting different physics from two different force fields on a givem molecule and you want to see what, if anything, changed between them in terms of which parameters were applied where. This does **not** compare numerical values, just the SMIRKS parameters used by different force fields, but could be extended to do so.

This example shows how to do it with the toolkit and how to do it with Interchange. Each should produce the same results, so use whichever is more convenient.

In [1]:
from openff.toolkit import Molecule, ForceField

molecule = Molecule.from_smiles("OC[C@@]5(O)C[C@@]31C[C@@H]5CC[C@H]1[C@]4(C)CCc2occc2[C@H]4CC3")

ff1 = ForceField("openff_unconstrained-2.2.1.offxml")
ff2 = ForceField("openff_unconstrained-2.3.0-rc2.offxml")


## Inspect SMIRKS-based parameter assignment as the toolkit does it

[`ForceField.label_molecules`](https://docs.openforcefield.org/projects/toolkit/en/stable/api/generated/openff.toolkit.typing.engines.smirnoff.ForceField.html#openff.toolkit.typing.engines.smirnoff.ForceField.label_molecules) reports which atoms (or bonds, angles, etc.) get which parameters. The return value is somewhat gnarly so let's decompose it.

In [2]:
ff1_assigned_parameters = ff1.label_molecules(molecule.to_topology())[0]
ff2_assigned_parameters = ff2.label_molecules(molecule.to_topology())[0]

# This is a dict of dicts; the high-level dict maps parameter handler names ('vdW', 'Bond', etc.)
# to dicts (type ValenceDict) containing parameters
# Each ValenceDict acts like a dict mapping tuples of atom indices to assigned parameter objects
# The keys are 1-length for vdW parameters, but would be longer for bonds, angles, torsions, etc.
# The parameter object here is vdWType, for others it would be BondType, etc.
ff1_assigned_parameters['vdW']

# since these are dicts, you could i.e. write them out to JSON for comparison later

ValenceDict(
  {(0,): <vdWType with smirks: [#8X2H1+0:1]  epsilon: 0.2094735324129 kilocalorie / mole  id: n19  rmin_half: 1.682099169199 angstrom  >,
   (1,): <vdWType with smirks: [#6X4:1]  epsilon: 0.1088406109251 kilocalorie / mole  id: n16  rmin_half: 1.896698071741 angstrom  >,
   (2,): <vdWType with smirks: [#6X4:1]  epsilon: 0.1088406109251 kilocalorie / mole  id: n16  rmin_half: 1.896698071741 angstrom  >,
   (3,): <vdWType with smirks: [#8X2H1+0:1]  epsilon: 0.2094735324129 kilocalorie / mole  id: n19  rmin_half: 1.682099169199 angstrom  >,
   (4,): <vdWType with smirks: [#6X4:1]  epsilon: 0.1088406109251 kilocalorie / mole  id: n16  rmin_half: 1.896698071741 angstrom  >,
   (5,): <vdWType with smirks: [#6X4:1]  epsilon: 0.1088406109251 kilocalorie / mole  id: n16  rmin_half: 1.896698071741 angstrom  >,
   (6,): <vdWType with smirks: [#6X4:1]  epsilon: 0.1088406109251 kilocalorie / mole  id: n16  rmin_half: 1.896698071741 angstrom  >,
   (7,): <vdWType with smirks: [#6X4:1]  

In [3]:
for handler_name in ff1_assigned_parameters:
    if handler_name in ['ToolkitAM1BCC', "NAGLCharges"]:
        continue

    print(f"looking at handler {handler_name}")
    # assume the keys are the same between these dicts
    these_parameters1 = ff1_assigned_parameters[handler_name]
    these_parameters2 = ff2_assigned_parameters[handler_name]

    for (
        (atom_indices1, parameter1),
        (atom_indices2, parameter2),
    ) in zip(
        these_parameters1.items(),
        these_parameters2.items(),
    ):
        assert atom_indices1 == atom_indices2, "Atom indices do not match!"

        # ParameterType.id is typically used for bookkeeping but that's not guaranteed
        # Compare SMIRKS instead since that's how the behavior is defined
        if parameter1.smirks != parameter2.smirks:
            print(f"\tDifference found applied to atom indices {atom_indices1}:")
            print(f"\t\tff1 applied SMIRKS: {parameter1.smirks}\n\t\tff2 applied SMIRKS: {parameter2.smirks}\n")

looking at handler Constraints
looking at handler Bonds
looking at handler Angles
	Difference found applied to atom indices (1, 0, 26):
		ff1 applied SMIRKS: [*:1]-[#8:2]-[*:3]
		ff2 applied SMIRKS: [#1:1]-[#8:2]-[*:3]

	Difference found applied to atom indices (2, 3, 29):
		ff1 applied SMIRKS: [*:1]-[#8:2]-[*:3]
		ff2 applied SMIRKS: [#1:1]-[#8:2]-[*:3]

	Difference found applied to atom indices (2, 4, 5):
		ff1 applied SMIRKS: [*;r5:1]1@[*;r5:2]@[*;r5:3]@[*;r5]@[*;r5]1
		ff2 applied SMIRKS: [*;r5:1]1@[*;r5:2]@[*;r5:3]@[*;r5]@[*;r5]~1

	Difference found applied to atom indices (2, 7, 6):
		ff1 applied SMIRKS: [*;r5:1]1@[*;r5:2]@[*;r5:3]@[*;r5]@[*;r5]1
		ff2 applied SMIRKS: [*;r5:1]1@[*;r5:2]@[*;r5:3]@[*;r5]@[*;r5]~1

	Difference found applied to atom indices (2, 7, 9):
		ff1 applied SMIRKS: [*;r6:1]~;@[*;r5;x4,*;r5;X4:2]~;@[*;r5;x2:3]
		ff2 applied SMIRKS: [*;r6:1]~;@[*;r5:2]~;@[*;r5;x2:3]

	Difference found applied to atom indices (4, 2, 7):
		ff1 applied SMIRKS: [*;r5:1]1@[*;r5:2]@[

In this case (for this molecule, for these force fields) the differences look relatively minor in that a few angle parameters were modified. While comparing other force fields and/or other molecules you might see more (or fewer) differences in other parameter handlers.

## Inspect parameter assignment in terms of how Interchange stores them

This route looks at what parameters are stored in an `Interchange` object, not what the toolkit expects would be stored. The results betwen these two methods should be the same for SMIRNOFF force fields since this ultimately relies on the toolkit for parameter assignment.

The process is similar: we apply each force field to our molecule of interest, iterate through the assigned parameters, and compare them.

Some more prose on this API can be found [here](https://docs.openforcefield.org/projects/interchange/en/stable/using/collections.html#).

In [4]:
interchange1 = ff1.create_interchange(molecule.to_topology())
interchange2 = ff2.create_interchange(molecule.to_topology())

for collection_name in interchange1.collections:
    if collection_name not in interchange2.collections:
        print(f"Collection {collection_name} not found in interchange2")
        continue

    print(f"looking at collection {collection_name}")
    these_parameters1 = interchange1.collections[collection_name]
    these_parameters2 = interchange2.collections[collection_name]

    for (
        (topology_key1, potential_key1),
        (topology_key2, potential_key2),
    ) in zip(
        these_parameters1.key_map.items(),
        these_parameters2.key_map.items(),
    ):
        assert topology_key1.atom_indices == topology_key2.atom_indices, "Atom indices do not match!"

        # these objects (`PotentialKey`s) are different from `ParameterType`s used in the toolkit
        # `.id` means something different here, and for SMIRNOFF force fields the SMIRKS string is stored here
        if potential_key1.id != potential_key2.id:
            print(f"\tDifference found applied to key {topology_key1}:")
            print(f"\t\tff1 applied SMIRKS: {potential_key1.id}\n\t\tff2 applied SMIRKS: {potential_key2.id}\n")

looking at collection Bonds
looking at collection Constraints
looking at collection Angles
	Difference found applied to key atom_indices=(1, 0, 26):
		ff1 applied SMIRKS: [*:1]-[#8:2]-[*:3]
		ff2 applied SMIRKS: [#1:1]-[#8:2]-[*:3]

	Difference found applied to key atom_indices=(2, 3, 29):
		ff1 applied SMIRKS: [*:1]-[#8:2]-[*:3]
		ff2 applied SMIRKS: [#1:1]-[#8:2]-[*:3]

	Difference found applied to key atom_indices=(2, 4, 5):
		ff1 applied SMIRKS: [*;r5:1]1@[*;r5:2]@[*;r5:3]@[*;r5]@[*;r5]1
		ff2 applied SMIRKS: [*;r5:1]1@[*;r5:2]@[*;r5:3]@[*;r5]@[*;r5]~1

	Difference found applied to key atom_indices=(2, 7, 6):
		ff1 applied SMIRKS: [*;r5:1]1@[*;r5:2]@[*;r5:3]@[*;r5]@[*;r5]1
		ff2 applied SMIRKS: [*;r5:1]1@[*;r5:2]@[*;r5:3]@[*;r5]@[*;r5]~1

	Difference found applied to key atom_indices=(2, 7, 9):
		ff1 applied SMIRKS: [*;r6:1]~;@[*;r5;x4,*;r5;X4:2]~;@[*;r5;x2:3]
		ff2 applied SMIRKS: [*;r6:1]~;@[*;r5:2]~;@[*;r5;x2:3]

	Difference found applied to key atom_indices=(4, 2, 7):
		ff1 app

These results look the same as before - cool!